# Titel der Analyse

Untertitel

Dein Name

## LLM-Setup

Um dieses Notizbuch mit Ollama zu verbinden, füge per Copy/Paste in der nächsten Code-Zelle einen validen `OLLAMA_API_KEY` ein und führe anschliessend diese Code-Zelle einmalig aus.

Wenn du dann Hilfe bei der Analyse benötigst, beispielsweise, um dir Python-Befehle oder Pandas erklären zu lassen, nutze jeweils in einer neuen Code-Zelle die Funktion

```python
chat("""
Hier dein (gern auch mehrzeiliger) Prompt ...
""")
```

Deinen eigenen, privaten API-Schlüssel erstellst du nach dem Login unter [ollama.com/settings/keys](https://ollama.com/settings/keys).

Konsultiere das Notizbuch [analyse-mit-Ollama-Details.ipynb](Ollama-Beispiele/analyse-mit-Ollama-Details.ipynb) im Ordner `Ollama-Beispiele`, wenn du mehr Informationen zu den Einstellungsmöglichkeiten, den verfügbaren Modellen oder dem lokalen Off-Line-Betrieb von Ollama suchst.

In [ ]:
OLLAMA_API_KEY="replace-me-with-a-valid-key"

max_chat_history = 7

model_name = "gemma4:31b"

priming = """
You are the best Python coder in the world.
You give short and informative answers.
You code like a student in 10th grade and prefer simple but readable solutions over compact solutions.
You know Pandas and all of Pandas data frame functionality inside out.
You live inside a Jupyter notebook, hence assume that the last line of each code block will be printed automatically.
You avoid using print() statements as last line of a code example but rely on Jupyter default printing capabilities.
You split Python code in short paragraphs appropriate for the Step-By-Step approach in Juypter Notebook Cells.
You try to encapsulate complex code examples into properly named functions with no side effects.
You are great at using and explaining pivot_tables, groupby statements and the build-in plotting capabilities of Pandas data frames.
You prefer to use Pandas data frames directly for plotting by appending .plot()
You avoid using matplotlib or seaborn explicitly wherever reasonable.
You always asume that a data frame named `df` exists.
You do NOT include example data for Pandas data frame named `df`
You do NOT overwrite the default data frame named `df` but keep it as golden source for raw data.
"""

##################################################################################################################
# Ab hier solltest du nichts mehr ändern müssen - führe einfach diese Zelle zu Beginn jeder Analyse einmalig aus #
##################################################################################################################

import ollama

client = ollama.Client(
    host="https://ollama.com",
    headers={'Authorization': 'Bearer ' + OLLAMA_API_KEY}
)

from IPython.display import display, Markdown
import pandas as pd
import json
import os

def load_conversation_log(filename = "llm_conversation_log.json"):
    """Liest ein .json-Datei, welche eine Aufzeichnung einer Unterhaltung enthält, ein."""
    global messages
    if os.path.exists(filename):
        with open(filename, 'r') as file:
            content = json.loads(file.read())
        if content[0].get("role") == "system":
            print("Loading", filename)
            messages = content

def save_conversation_log(filename = "llm_conversation_log.json"):
    """Schreibt die aktuelle Aufzeichnung der Unterhaltung in eine .json-Datei."""
    with open( filename , "w" ) as file:
        json.dump( messages , file )
    
def ask(prompt):
    """Einfacher Chat-Bot mit global spezifiziertem Client, Modell - ohne Historie"""
    global client
    global model_name
    response = client.generate(model=model_name, prompt=prompt)
    display(Markdown(response.response))

def remove_message(msg_list, n=1):
    """Entfernt die ältesten n einträge aus einer Unterhaltung"""
    #print("Cleanup - removing {} oldest message from chat history".format(n))
    return msg_list[0:1] + msg_list[(n+1):]
    
def chat(prompt):
    """Einfacher Chat-Bot mit global spezifiziertem Client, Modell und Historie"""
    global client
    global model_name
    global messages
    global max_chat_history
    
    messages.append({"role": "user", "content": prompt})

    if max_chat_history != 0:
        message_count = len(messages) - 1
        
        # remove exess chat history, if we have fixed size max_chat_history
        excess_messages = message_count - max_chat_history
        if max_chat_history > 0 and excess_messages > 0:
            messages = remove_message(messages, excess_messages)

        # if context window is (still) to long, we always remove old chat entries
        for _ in range(message_count + 1):
            try:
                response = client.chat(model=model_name, messages=messages)
            except ollama.ResponseError as e:
                if len(messages) > 1:
                    messages = remove_message(messages, 1)
                    continue
                raise
            except Exception as e:
                raise
    else:
        # no context lenght limit - expect your client to die eventually
        response = client.chat(model=model_name, messages=messages)
        
    messages.append({"role": "assistant", "content": response.message.content})
    save_conversation_log()
    display(Markdown(response.message.content))

def show_models_from_(output):
    """Gibt die Modelle von client.list() als Pandas Dataframe zurück"""
    df = pd.DataFrame([
        {
            "Model": entry.get("model", ""),
            "Size [GB]": round(int(entry.get("size", 0)) / 1024**3, 1)
        }
        for entry in output.get("models", [])
    ])

    if not df.empty:
        df = df.sort_values(by="Model", key=lambda s: s.str.lower())

    df = df.reset_index(drop=True)
    return df

messages = [
    {"role": "system", "content": priming}
]

MUST_HAVE_OBJECTS = ('ollama', 'client', 'chat', 'ask', 'max_chat_history', 'model_name', 'priming', 'messages')

llm_setup_errors = 0
for one_object in MUST_HAVE_OBJECTS:
    if one_object not in locals():
        llm_setup_errors += 1
        print("ERROR:", one_object, "nicht gefunden - bitte die zugehörige, vorhergehende Code-Zelle einmalig ausführen!")
        
assert llm_setup_errors == 0
assert model_name in list(show_models_from_(client.list())["Model"])

show_models_from_(client.list())

## Forschungsfragen

1. ...
2. ...
3. ...

## Daten einlesen

In [ ]:
load_conversation_log()

In [ ]:
chat("""
Wie liest man eine komma-separierte Datei in Python mit Pandas ein, 
wenn das Trennzeichen ein Semikolon ist? Was ändert sich, wenn das Trennzeichen ein Komma ist?
""")

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('pfad/zur/csv/datei.csv', sep=";")
df

## Daten vorverarbeiten

## Daten analysieren

## Daten visualisieren